# Generating jets and events with OmniX

This notebook loads the released OmniX model, samples JetClass-like jets and COLLIDE-2V-like
events with the sampling settings of the paper, and plots a few distributions.

Run `./checkpoints/download_checkpoints.sh` first. A GPU is needed.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

from humite.data.readers import PARTICLE_COLUMNS, condition_codes
from humite.data.tokenizer import EventTokenizer
from humite.models.generation import generate
from humite.pretrained import load_pretrained

module, data = load_pretrained("../checkpoints/omnix")
model = module.model
tokenizer = EventTokenizer(data, pad_length=data.max_particles)
codes = condition_codes(data)
print(sorted(codes, key=codes.get))

## Jets

`source=1` asks for a JetClass jet. Without a condition the jet type is the model's choice;
`condition=codes["TTBar"]` asks for a hadronic top jet. The paper's samples use temperature 1,
the default of `generate`.

In [ ]:
torch.manual_seed(0)
jets = generate(model, tokenizer, batch_size=512, source=1)
top_jets = generate(model, tokenizer, batch_size=512, source=1, condition=codes["TTBar"])

print(PARTICLE_COLUMNS)
print(jets["particles"].shape, jets["mask"].shape)

`particles` holds one row per particle with the columns printed above (pT in GeV, eta and phi in
the lab frame, the jet slot, charge, particle-flow type, and the impact parameters in cm), and
`mask` marks the particles that exist. `jets` is the plan the model generated first: the
four-momentum and the number of constituents of every jet slot. A JetClass jet fills slot 1.

In [ ]:
def jet_mass(sample):
    """Invariant mass of all particles of each sample, taken as massless."""
    p, m = sample["particles"].double().cpu(), sample["mask"].cpu()
    pt, eta, phi = (torch.where(m, p[..., i], torch.zeros_like(p[..., i])) for i in range(3))
    px, py, pz = (
        (pt * torch.cos(phi)).sum(1),
        (pt * torch.sin(phi)).sum(1),
        (pt * torch.sinh(eta)).sum(1),
    )
    energy = (pt * torch.cosh(eta)).sum(1)
    return (energy**2 - px**2 - py**2 - pz**2).clamp_min(0).sqrt().numpy()


fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
for sample, label in ((jets, "jet type not given"), (top_jets, "TTBar")):
    style = dict(histtype="step", density=True, label=label)
    axes[0].hist(sample["jets"]["jet_pt"][:, 0].cpu().numpy(), bins=40, range=(450, 1050), **style)
    axes[1].hist(jet_mass(sample), bins=40, range=(0, 300), **style)
    axes[2].hist(sample["mask"].sum(1).cpu().numpy(), bins=40, range=(0, 120), **style)
for ax, label in zip(axes, ("jet $p_T$ (GeV)", "jet mass (GeV)", "number of constituents")):
    ax.set_xlabel(label)
axes[0].legend()
plt.tight_layout()

## Events

`source=2` asks for a COLLIDE-2V event; the conditions 11-15 select the process. For events the
paper uses the out-of-jet exclusion: a particle outside every jet is not placed within R = 0.4
of a planned jet axis, as the jet algorithm would have clustered it into that jet.

In [ ]:
torch.manual_seed(0)
events = generate(
    model,
    tokenizer,
    batch_size=256,
    source=2,
    condition=codes["tt0123j_5f_ckm_LO_MLM_semiLeptonic"],
    out_of_jet_exclusion=0.4,
)

n_jets = (events["jets"]["n_const"] > 0).sum(1).cpu().numpy()
ht = events["jets"]["jet_pt"].sum(1).cpu().numpy()
n_particles = events["mask"].sum(1).cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
axes[0].hist(n_jets, bins=np.arange(-0.5, 20.5), histtype="step")
axes[1].hist(ht, bins=40, range=(0, 1500), histtype="step")
axes[2].hist(n_particles, bins=40, histtype="step")
for ax, label in zip(axes, ("number of jets", "$H_T$ (GeV)", "number of particles")):
    ax.set_xlabel(label)
plt.tight_layout()

One event in the eta-phi plane: every particle as a point with an area proportional to its pT,
coloured by its jet slot (grey: outside every jet).

In [ ]:
i = 0
p = events["particles"][i][events["mask"][i]].cpu().numpy()
pt, eta, phi, slot = p[:, 0], p[:, 1], p[:, 2], p[:, 3].astype(int)

plt.figure(figsize=(6, 4))
outside = slot == 0
plt.scatter(eta[outside], phi[outside], s=4 * pt[outside], color="0.6", linewidths=0)
plt.scatter(
    eta[~outside], phi[~outside], s=4 * pt[~outside], c=slot[~outside], cmap="tab20", linewidths=0
)
plt.xlabel(r"$\eta$")
plt.ylabel(r"$\phi$")
plt.xlim(-4, 4)
plt.ylim(-np.pi, np.pi)
plt.tight_layout()